###Create Schema

This notebook will create schemas for all 3 layers:<br>
Bronze layer -> raw<br>
Silver Layer -> struct<br>
Gold Layer -> prep<br>

In [0]:
dbutils.widgets.text("load_type", "")

In [0]:
load_type = dbutils.widgets.get("load_type")

if load_type == "static":
  load = "ref/config"
else:
  load = "config"

In [0]:
%run /Workspace/e-commerce/Parameter_setup

##Parameter Setup
This notebook is used to connect to storage account 

Parameter setup completed


####Schemas and Tables for Transactional Data

In [0]:
%sql

CREATE SCHEMA if not EXISTS adb_de_projects_001.ecom_raw

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.customers (
  customer_id STRING,
  customer_unique_id STRING,
  customer_zip_code_prefix INTEGER,
  customer_city STRING,
  customer_state STRING,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/customers';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.order_items (
  order_id STRING,
  order_item_id INTEGER,
  product_id STRING,
  seller_id STRING,
  shipping_limit_date TIMESTAMP,
  price DOUBLE,
  freight_value DOUBLE,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/order_items';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.order_payments (
  order_id STRING,
  payment_sequential INTEGER,
  payment_type STRING,
  payment_installments INTEGER,
  payment_value DOUBLE,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/order_payments';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.order_reviews (
    review_id STRING,
    order_id STRING,
    review_score STRING,
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date STRING,
    review_answer_timestamp STRING,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
)   
USING DELTA 
LOCATION '{raw_path}/order_reviews';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.orders (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/orders';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_raw.sellers (
    seller_id STRING,
    seller_zip_code_prefix INTEGER,
    seller_city STRING,
    seller_state STRING,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/sellers';
''').display()

In [0]:
%sql

CREATE SCHEMA if not EXISTS adb_de_projects_001.ecom_struct

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_struct.customers (
  customer_id STRING NOT NULL PRIMARY KEY,
  customer_unique_id STRING NOT NULL,
  customer_zip_code_prefix INTEGER,
  customer_city STRING,
  customer_state STRING,
  is_valid BOOLEAN,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{struct_path}/customers';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_struct.order_items (
  order_id STRING NOT NULL FOREIGN KEY references ecom_struct.orders(order_id), 
  order_item_id INTEGER NOT NULL PRIMARY KEY,
  product_id STRING NOT NULL FOREIGN KEY references ecom_ref_struct.products(product_id),
  seller_id STRING,
  shipping_limit_date TIMESTAMP,
  price DOUBLE,
  freight_value DOUBLE,
  is_valid BOOLEAN,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{struct_path}/order_items';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_struct.order_payments (
  order_id STRING NOT NULL FOREIGN KEY references ecom_struct.orders(order_id),
  payment_sequential INTEGER NOT NULL PRIMARY KEY,
  payment_type STRING,
  payment_installments INTEGER,
  payment_value DOUBLE,
  is_valid BOOLEAN,
  ingestion_date TIMESTAMP,
  pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{struct_path}/order_payments';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_struct.order_reviews (
    review_id STRING NOT NULL PRIMARY KEY,
    order_id STRING NOT NULL FOREIGN KEY references ecom_struct.orders(order_id),
    review_score STRING CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    is_valid BOOLEAN,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{struct_path}/order_reviews';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_struct.orders (
    order_id STRING NOT NULL PRIMARY KEY,
    customer_id STRING FOREIGN KEY references ecom_struct.customers(customer_id),
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    order_year INTEGER,
    order_month INTEGER,
    is_valid BOOLEAN,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{struct_path}/orders';
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS adb_de_projects_001.ecom_struct.sellers (
    seller_id STRING NOT NULL PRIMARY KEY,
    seller_zip_code_prefix STRING,
    seller_city STRING,
    seller_state STRING,
    is_valid BOOLEAN,
    ingestion_date TIMESTAMP,
    source_file_name STRING,
    pipeline_run_id STRING
)
USING DELTA
LOCATION '{struct_path}/sellers'
''').display()

DataFrame[]

In [0]:
%sql

CREATE SCHEMA if not EXISTS adb_de_projects_001.ecom_prep

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS ecom_prep.customers (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,  -- surrogate key
    customer_unique_id STRING,
    city STRING,
    state STRING,
    zip_code_prefix STRING,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    is_current BOOLEAN,
    start_date DATE,
    end_date DATE
)
USING DELTA
LOCATION '{prep_path}/customers'
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS ecom_prep.customers_map (
    customer_id STRING NOT NULL PRIMARY KEY,
    customer_unique_id STRING NOT NULL
)
USING DELTA
LOCATION '{prep_path}/customers_map'
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS ecom_prep.orders (
    order_item_id STRING NOT NULL,
    order_id STRING NOT NULL,
    customer_key BIGINT NOT NULL FOREIGN KEY references ecom_prep.customers(customer_key), 
    customer_unique_id STRING,
    seller_id STRING,
    seller_city STRING,
    seller_state STRING,
    product_id STRING,
    category_name_english STRING,
    order_status STRING,
    price DOUBLE,
    freight_value DOUBLE,
    total_revenue DOUBLE,
    payment_value DOUBLE,
    payment_installments INT,
    payment_type STRING,
    delivery_days INT,
    estimated_delivery_days INT,
    delivery_delay_days INT,
    order_purchase_date DATE,
    PRIMARY KEY (order_id, order_item_id)
)
USING DELTA
LOCATION '{prep_path}/orders'
''').display()

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS ecom_prep.reviews (
    review_id STRING NOT NULL PRIMARY KEY,
    order_id STRING NOT NULL,
    customer_key BIGINT NOT NULL FOREIGN KEY references ecom_prep.customers(customer_key), 
    review_score INT,
    review_comment_message STRING,
    sentiment STRING,
    positive_score FLOAT,
    neutral_score FLOAT,
    negative_score FLOAT,
    review_creation_date DATE,
    review_answer_timestamp TIMESTAMP,
    review_processed_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://e-commerce@datasets001.dfs.core.windows.net/prep/reviews'
''').display()

####Schemas for Referance/Static Data

In [0]:
%sql

CREATE SCHEMA if not EXISTS adb_de_projects_001.ecom_ref_raw

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_ref_raw.geolocation (
    geolocation_zip_code_prefix INTEGER,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city STRING,
    geolocation_state STRING,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/geolocation';
''')

DataFrame[]

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_ref_raw.product_category (
    product_category_name STRING,
    product_category_name_english STRING,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/product_category';
''')

DataFrame[]

In [0]:
spark.sql(f'''
CREATE TABLE if not exists adb_de_projects_001.ecom_ref_raw.products (
    product_id STRING,
    product_category_name STRING,
    product_name_lenght INTEGER,
    product_description_lenght INTEGER,
    product_photos_qty INTEGER,
    product_weight_g INTEGER,
    product_length_cm INTEGER,
    product_height_cm INTEGER,
    product_width_cm INTEGER,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
) 
USING DELTA 
LOCATION '{raw_path}/products';
''')

DataFrame[]

In [0]:
%sql

CREATE SCHEMA if not EXISTS adb_de_projects_001.ecom_ref_struct

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS adb_de_projects_001.ecom_ref_struct.geolocation (
    geolocation_zip_code_prefix STRING NOT NULL PRIMARY KEY,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city STRING,
    geolocation_state STRING,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
)
USING DELTA
LOCATION '{struct_path}/geolocation'
''')

DataFrame[]

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS adb_de_projects_001.ecom_ref_struct.products (
    product_id STRING NOT NULL PRIMARY KEY,
    product_category_name STRING,
    product_category_name_english STRING,
    product_name_length INTEGER,
    product_description_length INTEGER,
    product_photos_qty INTEGER,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm  DOUBLE,
    ingestion_date TIMESTAMP,
    pipeline_run_id STRING
)
USING DELTA
LOCATION '{struct_path}/products'
''')

DataFrame[]